# 05：在远程 CPU 上运行你的电路

**你将学到：**云端作业在另一个环境运行，有自己的 ID、状态和结果；量子电路仍是同一个 FlagQuantum 电路。

我们将一个 Bell 电路提交到九鼎。这是 CPU 实验，不是加速卡或性能加速演示。

[English edition](../notebooks/05_jiuding_jobs.ipynb)



## 1. 准备 workshop 环境

直接运行此设置单元，无需修改。它会找到你的个人结果目录。活动环境提供凭据和任务镜像，不要在 notebook 单元中填写密钥。在自己的电脑上使用时，请参考[环境准备](../SETUP.md)。

中英文版使用同一用户的个人结果目录和回执；切换语言不会自动创建新任务。不同用户的凭据和共享目录仍由平台管理。


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. 构造 Bell 电路并预测答案

`00` 和 `11` 的理想概率各为一半。先在本地检查。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
local = fq.run(circuit)
print("Local probabilities:", local.state.abs().square().tolist())


## 3. 选择远程 CPU

镜像提供任务中安装的软件。活动环境已预设镜像、项目和队列；在其他环境中，请使用你自己的九鼎配置。
**打开提交开关并运行提交单元，会创建真实 CPU 作业，消耗队列资源。**


In [ ]:
target = "jiuding:cpu"
image = os.getenv("JIUDING_IMAGE", "")
project = os.getenv("JIUDING_PROJECT", "")
queue = os.getenv("JIUDING_QUEUE")
submit_to_cloud = False


### 准备任务恢复文件（直接运行，无需修改）

这个折叠单元防止已保存任务被意外重复提交。它会检查电路与配置，并用 `fq.restore_job()` 恢复匹配的回执；它本身从不提交任务。
存储与凭据属于环境准备，不是量子计算操作。


In [ ]:
from flagquantum.core.ir import ensure_circuit_ir

receipt_path = OUTPUTS / "jiuding-bell-lesson.receipt.json"
guard_path = OUTPUTS / "jiuding-bell-lesson.submission.json"
request = {
    "circuit_hash": ensure_circuit_ir(circuit).content_hash,
    "target": target,
    "shots": None,
    "image": image,
    "project": project,
    "queue": queue,
}
job = None
if submit_to_cloud:
    if not image or not project:
        raise RuntimeError("Ask your instructor to configure the Jiuding image and project; nothing submitted.")
    if receipt_path.exists():
        saved = json.loads(guard_path.read_text())
        if any(saved.get(key) != value for key, value in request.items()):
            raise RuntimeError("Saved task belongs to another experiment. Keep its files and use a new receipt filename.")
        job = fq.restore_job(receipt_path)
    else:
        try:
            with guard_path.open("x") as stream:
                json.dump(request, stream, indent=2)
        except FileExistsError as exc:
            raise RuntimeError("An earlier attempt has no receipt. Reconcile it on the platform; do not resubmit.") from exc


## 4. 通过 FlagQuantum 提交

`fq.submit()` 返回时不会等待任务执行完成。立即调用 `job.save()` 保存回执，就能在内核重启后恢复同一个任务。
重复运行此 notebook 时，会恢复记录中的任务，而不是再次提交。

若修改了电路或提交配置，恢复检查不会把旧任务当成新实验。请遵循提示，为新的实验选择新的回执文件，不要删除仍需保留的记录。


In [ ]:
if submit_to_cloud and job is None:
    job = fq.submit(
        circuit, target=target, image=image, project=project, queue=queue,
    )
    print("Job ID:", job.id)
    job.save(receipt_path)
elif job is not None:
    print("Restored job:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. 查询进度

这里执行一次只读查询。`queued` 表示排队，`running` 表示正在执行，`succeeded` 表示平台报告已完成。
结果交付是另一个环节，仍可能延迟或暂不可用。


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. 短暂等待并读取结果

`job.wait()` 将轮询状态和获取结果组合起来。超时不会停止任务，可以稍后重新运行此单元。
也可以在状态为 `succeeded` 后调用 `job.result()`。
如果远程任务失败，不会用本地结果冒充远程结果。


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    import torch

    torch.testing.assert_close(result.state, local.state)
    print("Remote probabilities:", result.state.abs().square().tolist())
    print("The remote CPU calculation matches the local calculation.")


## 带回家使用

离开 Jupyter 后仍然使用相同的标准 API。这个简短示例每次被主动执行都会创建一个新的真实任务；请先配置镜像、项目和队列。

```python
import os
import flagquantum as fq

circuit = fq.Circuit(2).h(0).cx(0, 1)
job = fq.submit(
    circuit, target="jiuding:cpu", image=os.environ["JIUDING_IMAGE"],
    project=os.environ["JIUDING_PROJECT"], queue=os.getenv("JIUDING_QUEUE"),
)
job.save("bell.receipt.json")
print(job.id, job.status())
```

恢复已保存任务：`job = fq.restore_job("bell.receipt.json")`。完成后可查看 `job.result().state`。

**解释结果：**哪些调用负责提交、查询、等待和获取结果？远程运行一个小电路是否能证明性能加速？
完整 Python 程序和加速器选项见[云任务说明](../CLOUD_JOBS.md)。
